In [ ]:
import torch
import torch.nn as nn
from torch.nn.parameter import Parameter
import torch.nn.functional as F
import torch.utils.data as data

import pandas as pd

#ANFIS Structure and Functions

##Functions

In [ ]:
gaussian2_params = ['mu', 'sigma']
def gaussian2(x, p):
    p[:, :, 1] = torch.where(p[:, :, 1] == 0, torch.tensor(1e-6), p[:, :, 1])
    return torch.exp(-0.5 * torch.pow((x.unsqueeze(x.dim()) - p[:, :, 0])/p[:, :, 1], 2))

In [ ]:
gaussian3_params = ['mu', 'sigma', 'f']
def gaussian3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x.unsqueeze(x.dim()) - p[:, :, 0])/p[:, :, 1], 2))

In [ ]:
def weighted_linear(x, c, w):
    return (x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1]).mul(w)

In [ ]:
def sum(x):
    return torch.sum(x, dim=-1)

##ANFIS Layers

###Fuzzify Layer

In [ ]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of initial rules
        mf: function used as membership function
        params: list with parameter names
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly

    Other attributes:
        premises: array with the premises parameters used in each node of this layer
    '''
    def __init__(self, input_size, init_rules=3, mf=gaussian2, params=['mu', 'sigma'], x_train=[]):
        super(FuzzifyLayer, self).__init__()
        self.input_size = input_size
        self.init_rules = init_rules
        self.mf = mf
        self.params = params
        if x_train == []:
            prems = 2 * torch.rand(input_size, init_rules, len(params), dtype=torch.double) - 1
            prems[:,:,1] = (prems[:,:,1] + 1)/2 #Sigma stays positive
            self.premises = Parameter(prems, requires_grad=True)
        else:
            self.premises = Parameter(self.init_premises(x_train), requires_grad=True)

    def init_premises(self, x_train):
        premises = torch.zeros(self.input_size, self.init_rules, len(self.params), dtype=torch.double)
        if self.init_rules > 1:
            min = torch.min(x_train, dim=0).values
            max = torch.max(x_train, dim=0).values
            stp = (max - min) / (self.init_rules - 1)
            for i in range(self.input_size):
                h = torch.arange(min[i], max[i] + stp[i], stp[i])
                premises[i, :, 0] = h[:self.init_rules]
                premises[i, :, 1] = stp[i]/2
                if (len(self.params) == 3): premises[i, :, 2] = 2
        else:
            for i in range(self.input_size):
                premises[i, :, 0] = torch.mean(x_train[:, i])
                premises[i, :, 1] = torch.std(x_train[:, i])
                if (len(self.params) == 3): premises[i, :, 2] = 2
        return premises

    def forward(self, x):
        return self.mf(x, self.premises)

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_size):
            print(f"    x{i} parameters:")
            for j in range(self.init_rules):
                print(f"        rule {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]

###Firing Levels Layer

In [ ]:
class FiringLevelsLayer(nn.Module):
    '''
    Class used to calculate firing levels and normalize them
    '''
    def __init__(self):
        super(FiringLevelsLayer, self).__init__()

    def forward(self, x, rules):
        if rules == 1:
            fls = x.prod(dim=x.dim()-2)
            fls = fls/torch.where(fls == 0, torch.tensor(1e-6), fls)
        else:
            sum = torch.sum(x.prod(dim=x.dim()-2), dim=1, keepdim=True)
            sum[sum == 0] = 1
            fls = x.prod(dim=x.dim()-2)/sum
        return fls

###Consequents Layer

In [ ]:
class ConsequentLayer(nn.Module):
    '''
    Class that represents the fourth layer (consequent layer) of an ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of init_rules
        function : function used as a consequent function

    Other attributes:
        consequents: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, init_rules=3, function=weighted_linear):
        super(ConsequentLayer, self).__init__()
        self.init_rules = init_rules
        self.input_size = input_size
        self.function = function
        self.consequents = Parameter(2 * torch.rand(init_rules, input_size + 1, dtype=torch.double) - 1, requires_grad=True)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    @property
    def consequents_structure(self):
        print("Consequents Structure:")
        for i in range(self.init_rules):
            print(f"    rule {i + 1} consequent parameters: {self.consequents[i]}")

###Output Layer

In [ ]:
class OutputLayer(nn.Module):
    '''
    Class that represents the fifth layer (output layer) of an ANFIS
    '''
    def __init__(self, function=sum):
        super(OutputLayer, self).__init__()
        self.function = function

    def forward(self, x):
        return self.function(x)

##ANFIS

In [ ]:
class Type3ANFIS(nn.Module):
    '''
    Class that represents a type3 ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of initial rules
        cf: function used as consequent function
        mf: function used as membership function
        of: function used to join the outputs of each of the rules
        mf_params: list with MF parameters namesr
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly
    '''
    def __init__(self, input_size, init_rules=3, cf=weighted_linear, mf=gaussian2, of=sum, mf_params=['mu', 'sigma'], x_train=[]):
        super(Type3ANFIS, self).__init__()
        self.input_size = input_size
        self.mf_params = mf_params
        if x_train == []:
            self.fuzzify_layer = FuzzifyLayer(input_size, init_rules, mf, mf_params)
        else:
            self.fuzzify_layer = FuzzifyLayer(input_size, init_rules, mf, mf_params, x_train)
        self.firing_levels_layer = FiringLevelsLayer()
        self.consequent_layer = ConsequentLayer(input_size, init_rules, cf)
        self.output_layer = OutputLayer(of)

    def forward(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.firing_levels_layer(output, self.rules))
        output = self.output_layer(output)
        return output

    def firing_levels(self, x):
        x = self.fuzzify_layer(x)
        fl = x.prod(dim=x.dim()-2)
        return fl

    def outputs_by_rule(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.firing_levels_layer(output, self.rules))
        return output

    @property
    def rules(self):
        return self.consequents.shape[0]

    @property
    def premises_structure(self):
        self.fuzzify_layer.premises_structure

    @property
    def premises(self):
        return self.fuzzify_layer.premises

    def set_premises(self, premises):
        self.fuzzify_layer.premises = premises

    @property
    def consequents_structure(self):
        self.consequent_layer.consequents_structure

    @property
    def consequents(self):
        return self.consequent_layer.consequents

    def set_consequents(self, consequents):
        self.consequent_layer.consequents = consequents

#OLS

In [ ]:
def OLS(ANFISmodel, loader, y, epochs, ):
    ep = 0
    while (ep < epochs):
        #---CONSEQUENTS ESTIMATION---

        #needed tensors for consequents calculation
        x_train = test_loader.dataset.tensors[0]
        y_train = test_loader.dataset.tensors[1] #targets
        new_consequents = torch.zeros_like(ANFISmodel.consequents.data)
        xe = torch.cat([x_train, torch.ones(x_train.shape[0], 1)], dim=1) #extended x_train

        #normalized firing levels
        if ANFISmodel.rules == 1: #if there is one rule (Nx1 tensor), torch normalization doesnt work correctly
            fls = ANFISmodel.firing_levels(x_train)
            fls = fls/fls
        else:
            #fls = torch.nn.functional.normalize(ANFISmodel.firing_levels(x_train), p=1, dim=-1) #normalized firing_levels
            sum = torch.sum(ANFISmodel.firing_levels(x_train), dim=1, keepdim=True)
            sum[sum == 0] = 1
            fls = ANFISmodel.firing_levels(x_train)/sum

        #iterates by rule
        for i in range(ANFISmodel.rules):
            #firing_levels diagonal matrix
            fls_diag = torch.diag(fls[:, i])

            #new consequents
            #new_consequents[i] = torch.inverse(xe.t() @ fls_diag @ xe) @ xe.t() @ fls_diag @ y_train
            new_consequents[i], _, _, _ = torch.linalg.lstsq(xe.t() @ fls_diag @ xe, xe.t() @ fls_diag @ y_train)

        ANFISmodel.set_consequents(Parameter(new_consequents, requires_grad=True))

        #---PREMISES ESTIMATION---

        #needed tensors for consequents calculation
        pred = ANFISmodel(x_train) #models prediction

        #mse and premises gradients calculation with pytorchs backward function
        mse = nn.functional.mse_loss(pred, y_train)
        mse.backward()

        #alpha calculation (with premises gradients obtained before with mse.backward())
        alpha = y / torch.sqrt(torch.sum(torch.pow(ANFISmodel.premises.grad, 2)))

        #premises extracted
        current_premises = ANFISmodel.premises.data
        vs = current_premises[:,:,0].t()
        sigmas = current_premises[:,:,1].t()
        new_vs = torch.zeros_like(vs)
        new_sigmas = torch.zeros_like(sigmas)

        #iterates by rule
        for i in range(ANFISmodel.rules):
            #mu
            A = 4*alpha*(1/torch.pow(sigmas[i], 2))*(x_train - vs[i])
            wk = fls[:,i].unsqueeze(0).t()
            fk = ANFISmodel.outputs_by_rule(x_train)[:,i]
            zk = ((fk-pred)*(y_train-pred)).unsqueeze(0).t()

            new_vs[i] = torch.sum(A*wk*zk, dim=0)

            #sigma
            B = 4*alpha*(1/torch.pow(sigmas[i], 3))*torch.pow((x_train - vs[i]), 2)

            new_sigmas[i] = torch.sum(B*wk*zk, dim=0)

        #Premises update
        current_premises[:, :, 0] += new_vs.t()
        current_premises[:, :, 1] += new_sigmas.t()

        #Parameters set on ANFISmodel
        ANFISmodel.set_premises(Parameter(current_premises, requires_grad=True))

        ep = ep+1

#SONFIS

##GrowNet

In [ ]:
def GrowNet(ANFISmodel, loader, ages, Ngrow, dGrow):
    first_batch = True
    for x_batch, y_batch in loader:
        #Max firing levels are obtenined
        firing_levels = ANFISmodel.firing_levels(x_batch)
        max_fl = torch.max(firing_levels, dim=1)

        #Boolean mask to filter the samples
        dGrow_mask = (max_fl.values <= dGrow**ANFISmodel.input_size)

        #Necesary tensors are defined on the first iteration
        if first_batch:
            bad_samples = torch.tensor([])
            best_bs_rules = torch.tensor([], dtype=torch.int)
            first_batch = False

        #The samples are extracted by concatenating tensors (which are filtered by the mask)
        bad_samples = torch.cat((bad_samples, x_batch[dGrow_mask]), dim=0)
        best_bs_rules = torch.cat((best_bs_rules, max_fl.indices[dGrow_mask]), dim=0)

    #Ngrow parameter filter
    unique_rules, counts = torch.unique(best_bs_rules, return_counts=True)
    Ngrow_mask = (counts > Ngrow)

    indices_to_keep = torch.isin(best_bs_rules, unique_rules[Ngrow_mask]).nonzero().squeeze()

    bad_samples = bad_samples[indices_to_keep]
    best_bs_rules = best_bs_rules[indices_to_keep]

    #return False if ANFISmodel is not modified
    if bad_samples.size(0) == 0:
        return False, ages

    #a list of masks called "rules" is created to calculate the necessary means and stds
    rules = [best_bs_rules == value for value in torch.unique(best_bs_rules)]

    #means and stds by rule are calculated
    means = torch.stack([(bad_samples[rule].mean(dim=0)) for rule in rules])
    stds = torch.stack([(bad_samples[rule].std(dim=0)) for rule in rules])

    #Premises and consequents modifications to add a new rule
    new_premises = torch.stack([means.t(), stds.t()], dim=2)
    ANFISmodel.set_premises(Parameter(torch.cat([ANFISmodel.premises.data, new_premises], dim=1), requires_grad=True))

    new_consequents = 2 * torch.rand(new_premises.size(1), ANFISmodel.input_size + 1) - 1
    ANFISmodel.set_consequents(Parameter(torch.cat([ANFISmodel.consequents.data, new_consequents]), requires_grad=True))

    #New rules age
    ages = torch.cat([ages, torch.zeros(new_premises.shape[1], dtype=torch.bool)])

    #return True if ANFISmodel is modified
    return True, ages

##Split Sub-network

In [ ]:
def SplitSubNetwork(ANFISmodel, loader, ages, Nsplit, eSplit):
    first_batch = True
    for x_batch, y_batch in loader:
        #Max firing levels are obtenined
        firing_levels = ANFISmodel.firing_levels(x_batch)
        max_fl = torch.max(firing_levels, dim=1)

        #Necesary tensors are defined on the first iteration
        if first_batch:
            samples = torch.tensor([])
            samples_output = torch.tensor([])
            best_rules = torch.tensor([], dtype=torch.int)
            first_batch = False

        #The best rules are extracted by concatenating tensors
        samples = torch.cat((samples, x_batch), dim=0)
        samples_output = torch.cat((samples_output, y_batch), dim=0)
        best_rules = torch.cat((best_rules, max_fl.indices), dim=0)

    #Nsplit parameter filter
    unique_rules, counts = torch.unique(best_rules, return_counts=True)
    Nsplit_mask = (counts > Nsplit)

    indices_to_keep = torch.isin(best_rules, unique_rules[Nsplit_mask]).nonzero().squeeze()

    samples = samples[indices_to_keep]
    samples_output = samples_output[indices_to_keep]
    best_rules = best_rules[indices_to_keep]

    #the rules from best_rules tensor are extracted
    unique_rules = torch.unique(best_rules)

    #MSE is calculated by rule
    mse_values = torch.stack([torch.pow((samples_output[best_rules == rule] - ANFISmodel(samples[best_rules == rule])), 2).mean(dim=0) for rule in unique_rules])

    #eSplit parameter filter
    eSplit_mask = (mse_values > eSplit)

    #return Flase if ANFISmodel is not modified
    if ((samples.shape[0] == 0) | (unique_rules[eSplit_mask].shape[0] == 0)):
        return False, ages

    #loop to split each rule one by one and generate the new ones
    new_premises = ANFISmodel.premises.data #new_premises starts being a copy of the current premises
    new_consequents = ANFISmodel.consequents.data #same thing with consequents
    for rule in list(torch.flip(unique_rules[eSplit_mask], [0]).long()): #the iteration is performed on a list with the rules in descending order
        #the selected premise is extracted from the new_premises tensor and placed into the to_split tensor
        new_premises = torch.cat([new_premises[:, :rule, :], new_premises[:, rule+1:, :]], dim=1)
        to_split = ANFISmodel.premises.data[:, rule:rule+1, :].clone()

        #the new ones are generated
        split1 = torch.cat([(to_split[:,:,0] - to_split[:,:,1]/2).unsqueeze(1), (to_split[:,:,1]/2).unsqueeze(1)], dim=2)
        split2 = torch.cat([(to_split[:,:,0] + to_split[:,:,1]/2).unsqueeze(1), (to_split[:,:,1]/2).unsqueeze(1)], dim=2)

        #both are inserted on the new premises tensor
        new_premises = torch.cat([new_premises, torch.cat([split1, split2], dim=1)], dim=1)

        #the corresponding consequent is taken away
        new_consequents = torch.cat([new_consequents[:rule, :], new_consequents[rule+1:, :]], dim=0)

        #two new consequents are added
        new_consequents = torch.cat([new_consequents, 2 * torch.rand(2, ANFISmodel.input_size + 1) - 1], dim=0)

        #same with ages tensor
        ages = torch.cat([ages[:rule], ages[rule+1:]]) #the corresponding rule is taken away
        ages = torch.cat([ages, torch.zeros(2, dtype=torch.bool)]) #the new ones are added

    #after the loop, the new parameters are set
    ANFISmodel.set_premises(Parameter(new_premises, requires_grad=True))
    ANFISmodel.set_consequents(Parameter(new_consequents, requires_grad=True))

    #return True if ANFISmodel is modified
    return True, ages

##VanishNet

In [ ]:
def VanishSubNetwork(ANFISmodel, loader, ages, Nvanish, lVanish, last_best_rules):
    first_batch = True
    for x_batch, y_batch in loader:
        #Max firing levels are obtenined
        firing_levels = ANFISmodel.firing_levels(x_batch)
        max_fl = torch.max(firing_levels, dim=1)

        #Necesary tensors are defined on the first iteration
        if first_batch:
            best_rules = torch.tensor([], dtype=torch.int)
            first_batch = False

        #the best_rules by sample are extracted
        best_rules = torch.cat((best_rules, max_fl.indices), dim=0)

    #Important info for net operations
    unique_rules, counts = torch.unique(best_rules, return_counts=True)
    all_rules = torch.arange(ANFISmodel.rules)

    #tensor with te amounts of samples modeled for each rule (including those who dont model any sample)
    total_counts = torch.zeros(ANFISmodel.rules, dtype=torch.int64)
    total_counts[unique_rules] = counts

    #if there is no changes with the modeled samples or there is no last_best_rules (first iteration of VanishNet operator)
    if torch.equal(best_rules, last_best_rules) | torch.equal(last_best_rules, torch.tensor([-1])):
        #ages update
        ages += 1
    else:
        #last best rules are inspected (only the ones that appeared in the current iteration)
        last_unique_rules, last_counts = torch.unique(last_best_rules, return_counts=True)
        last_total_counts = torch.zeros(ANFISmodel.rules, dtype=torch.int64)
        last_total_counts[last_unique_rules[last_unique_rules < ANFISmodel.rules]] = last_counts[last_unique_rules < ANFISmodel.rules]

        #rules categorization
        improved_rules = all_rules[last_total_counts < total_counts]
        not_improved_rules = all_rules[last_total_counts >= total_counts]

        #ages update
        ages[improved_rules] = 0
        ages[not_improved_rules] += 1

    #lVanish and Nvanish filters
    mask = ((ages > lVanish) & (total_counts < Nvanish))
    rules_to_eliminate = all_rules[mask]

    if torch.equal(rules_to_eliminate, torch.tensor([], dtype=torch.int64)):
        return False, best_rules, ages

    #Parameters update (the rules are eliminated)
    new_premises = ANFISmodel.premises.data
    new_consequents = ANFISmodel.consequents.data
    for rule in rules_to_eliminate:
        new_premises = torch.cat([new_premises[:, :rule, :], new_premises[:, rule+1:, :]], dim=1)
        new_consequents = torch.cat([new_consequents[:rule, :], new_consequents[rule+1:, :]], dim=0)
        ages = torch.cat([ages[:rule], ages[rule+1:]]) #the corresponding rule is taken away

    #New parameters are set
    ANFISmodel.set_premises(Parameter(new_premises, requires_grad=True))
    ANFISmodel.set_consequents(Parameter(new_consequents, requires_grad=True))

    #The best rules tensor is returned for next iterations, also the ages
    return True, best_rules, ages

##SONFIS Implementation

In [ ]:
def SONFIS(ANFISmodel, loader, y, Ngrow, dGrow, Nsplit, eSplit, Nvanish, lVanish):
    #necessary initial values
    model_updated = True
    ages = torch.zeros(ANFISmodel.rules)
    last_best_rules = torch.tensor([-1], dtype=torch.int)

    print("\npre OLS premises", ANFISmodel.premises.data)
    print("pre OLS consequnets", ANFISmodel.consequents.data)

    while model_updated:
        #procedure
        OLS(ANFISmodel, loader, y)
        print("\npost OLS premises", ANFISmodel.premises.data)
        print("post OLS consequents", ANFISmodel.consequents.data)
        did_Grow, ages = GrowNet(ANFISmodel, loader, ages, Ngrow, dGrow)
        print("\ndid_Grow: ", did_Grow)
        if not did_Grow:
            did_Split, ages = SplitSubNetwork(ANFISmodel, loader, ages, Nsplit, eSplit)
            print("did_Split: ", did_Split)
        else:
            did_Split = False
            print("did_Split: ", did_Split)
        did_Vanish, last_best_rules, ages = VanishSubNetwork(ANFISmodel, loader, ages, Nvanish, lVanish, last_best_rules)
        print("did_Vanish: ", did_Vanish)

        print("\n-------------------------------------------------")
        print("\nRules amount:", ANFISmodel.rules)

        #did the model change?
        model_updated = did_Grow | did_Split | did_Vanish

        print("\npre OLS premises", ANFISmodel.premises.data)
        print("pre OLS consequents", ANFISmodel.consequents.data)

    OLS(ANFISmodel, loader, y)
    print("\npost OLS premises", ANFISmodel.premises.data)
    print("post OLS consequents", ANFISmodel.consequents.data)

#DataLoader

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
data_path = "/content/drive/MyDrive/Práctica Industrial/Experimentación inicial/Matlab_Example/Agua150321.xlsx"

In [ ]:
data_frame = pd.read_excel(data_path, sheet_name='pred')

In [ ]:
columns = [f'x{i}' for i in range(data_frame.shape[1] - 1)] + ['y']
data_frame.columns = columns

In [ ]:
data_frame

,x0,x1,x2,x3,x4,y
0,18.5,19.0,20.1,20.6,0.0,18.0
1,18.0,18.5,19.0,20.1,0.0,18.0
2,18.0,18.0,18.5,19.0,0.0,18.0
3,18.0,18.0,18.0,18.5,0.0,17.6
4,17.6,18.0,18.0,18.0,0.0,17.6
...,...,...,...,...,...,...
6439,26.2,26.9,27.0,26.0,0.0,24.5
6440,24.5,26.2,26.9,27.0,0.0,23.9
6441,23.9,24.5,26.2,26.9,0.0,23.2
6442,23.2,23.9,24.5,26.2,0.0,23.5


In [ ]:
X = torch.tensor(data_frame.iloc[:, :-1].values)
Y = torch.tensor(data_frame.iloc[:, -1].values)

In [ ]:
x_train = X[:6205, :]
x_test = X[6205:, :]
y_train = Y[:6205]
y_test = Y[6205:]

In [ ]:
train_loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 8)
test_loader = data.DataLoader(data.TensorDataset(x_test, y_test), batch_size = 8)

#Testing for Problems Resolving

In [ ]:
#Redundant
x_train = train_loader.dataset.tensors[0]
input_dim = x_train.shape[1]
input_dim

5

In [ ]:
model = Type3ANFIS(input_dim, init_rules=1)

In [ ]:
model.premises

Parameter containing:
tensor([[[ 0.7952,  0.4280]],

        [[ 0.9983,  0.0810]],

        [[ 0.6127,  0.6724]],

        [[ 0.1838,  0.6478]],

        [[-0.3213,  0.5364]]], dtype=torch.float64, requires_grad=True)

In [ ]:
model.consequents

Parameter containing:
tensor([[-0.2481, -0.6953, -0.4286, -0.3642,  0.3701, -0.4661]],
       dtype=torch.float64, requires_grad=True)

In [ ]:
#---CONSEQUENTS ESTIMATION---

#needed tensors for consequents calculation
x_train = train_loader.dataset.tensors[0]
y_train = train_loader.dataset.tensors[1] #targets
new_consequents = torch.zeros_like(model.consequents.data)
xe = torch.cat([x_train, torch.ones(x_train.shape[0], 1)], dim=1) #extended x_train

In [ ]:
if model.rules == 1: #if there is one rule (Nx1 tensor), torch normalization doesnt work correctly
    fls = model.firing_levels(x_train)
    fls = fls/torch.where(fls == 0, torch.tensor(1e-6), fls)

else:
    fls = torch.nn.functional.normalize(model.firing_levels(x_train), p=1, dim=-1) #normalized firing_levels

RuntimeError: a view of a leaf Variable that requires grad is being used in an in-place operation.

In [ ]:
unique_fl, counts = torch.unique(fls, return_counts=True)

In [ ]:
unique_fl

tensor([0.], dtype=torch.float64, grad_fn=<Unique2Backward0>)

In [ ]:
#iterates by rule
for i in range(model.rules):
    #firing_levels diagonal matrix
    fls_diag = torch.diag(fls[:, i])

    #new consequents
    new_consequents[i], _, _, _ = torch.linalg.lstsq(xe.t() @ fls_diag @ xe, xe.t() @ fls_diag @ y_train)


In [ ]:
new_consequents

tensor([[0., 0., 0., 0., 0., 0.]], dtype=torch.float64, grad_fn=<CopySlices>)

In [ ]:
model.set_consequents(Parameter(new_consequents, requires_grad=True))

In [ ]:
model.consequents

Parameter containing:
tensor([[0., 0., 0., 0., 0., 0.]], dtype=torch.float64, requires_grad=True)

In [ ]:
x_train

tensor([[18.5000, 19.0000, 20.1000, 20.6000,  0.0000],
        [18.0000, 18.5000, 19.0000, 20.1000,  0.0000],
        [18.0000, 18.0000, 18.5000, 19.0000,  0.0000],
        ...,
        [65.7000, 66.6000, 66.6000, 67.1000,  0.0000],
        [63.2000, 65.7000, 66.6000, 66.6000,  0.0000],
        [47.9000, 51.4000, 55.8000, 58.1000,  0.0000]], dtype=torch.float64)

In [ ]:
pred = model(x_train)
pred

RuntimeError: a view of a leaf Variable that requires grad is being used in an in-place operation.

In [ ]:
mse = nn.functional.mse_loss(pred, y_train)
mse.backward()

In [ ]:
alpha = 0.01 / torch.sqrt(torch.sum(torch.pow(model.premises.grad, 2)))

In [ ]:
current_premises = model.premises.data
vs = current_premises[:,:,0].t()
sigmas = current_premises[:,:,1].t()
new_vs = torch.zeros_like(vs)
new_sigmas = torch.zeros_like(sigmas)

In [ ]:
for i in range(model.rules):
    #mu
    A = 4*alpha*(1/torch.pow(sigmas[i], 2))*(x_train - vs[i])
    wk = fls[:,i].unsqueeze(0).t()
    fk = model.outputs_by_rule(x_train)[:,i]
    zk = ((fk-pred)*(y_train-pred)).unsqueeze(0).t()

    new_vs[i] = torch.sum(A*wk*zk, dim=0)

    #sigma
    B = 4*alpha*(1/torch.pow(sigmas[i], 3))*torch.pow((x_train - vs[i]), 2)

    new_sigmas[i] = torch.sum(B*wk*zk, dim=0)

In [ ]:
#Premises update
current_premises[:, :, 0] += new_vs.t()
current_premises[:, :, 1] += new_sigmas.t()

In [ ]:
current_premises

tensor([[[nan, nan]],

        [[nan, nan]],

        [[nan, nan]],

        [[nan, nan]],

        [[nan, nan]]], dtype=torch.float64, grad_fn=<CopySlices>)

In [ ]:

'''
#---PREMISES ESTIMATION---

#needed tensors for consequents calculation
pred = model(x_train) #models prediction

#mse and premises gradients calculation with pytorchs backward function
mse = nn.functional.mse_loss(pred, y_train)
mse.backward()

#print(f"PREMISES GRAD POST BACKWARD PASS: {ANFISmodel.premises.grad}")

#alpha calculation (with premises gradients obtained before with mse.backward())
alpha = y / torch.sqrt(torch.sum(torch.pow(model.premises.grad, 2)))

#premises extracted
current_premises = model.premises.data
vs = current_premises[:,:,0].t()
sigmas = current_premises[:,:,1].t()
new_vs = torch.zeros_like(vs)
new_sigmas = torch.zeros_like(sigmas)

    #iterates by rule
for i in range(model.rules):
    #mu
    A = 4*alpha*(1/torch.pow(sigmas[i], 2))*(x_train - vs[i])
    wk = torch.nn.functional.normalize(fls, p=1, dim=-1)[:,i].unsqueeze(0).t()
    fk = model.outputs_by_rule(x_train)[:,i]
    zk = ((fk-pred)*(y_train-pred)).unsqueeze(0).t()

    new_vs[i] = torch.sum(A*wk*zk, dim=0)

    #sigma
    B = 4*alpha*(1/torch.pow(sigmas[i], 3))*torch.pow((x_train - vs[i]), 2)

    new_sigmas[i] = torch.sum(B*wk*zk, dim=0)

#Premises update
current_premises[:, :, 0] += new_vs.t()
current_premises[:, :, 1] += new_sigmas.t()
'''

'\n#---PREMISES ESTIMATION---\n\n#needed tensors for consequents calculation\npred = model(x_train) #models prediction\n\n#mse and premises gradients calculation with pytorchs backward function\nmse = nn.functional.mse_loss(pred, y_train)\nmse.backward()\n\n#print(f"PREMISES GRAD POST BACKWARD PASS: {ANFISmodel.premises.grad}")\n\n#alpha calculation (with premises gradients obtained before with mse.backward())\nalpha = y / torch.sqrt(torch.sum(torch.pow(model.premises.grad, 2)))\n\n#premises extracted\ncurrent_premises = model.premises.data\nvs = current_premises[:,:,0].t()\nsigmas = current_premises[:,:,1].t()\nnew_vs = torch.zeros_like(vs)\nnew_sigmas = torch.zeros_like(sigmas)\n\n    #iterates by rule\nfor i in range(model.rules):\n    #mu\n    A = 4*alpha*(1/torch.pow(sigmas[i], 2))*(x_train - vs[i])\n    wk = torch.nn.functional.normalize(fls, p=1, dim=-1)[:,i].unsqueeze(0).t()\n    fk = model.outputs_by_rule(x_train)[:,i]\n    zk = ((fk-pred)*(y_train-pred)).unsqueeze(0).t()\n\n 

In [ ]:
new_consequents

tensor([[0., 0., 0., 0., 0., 0.]], dtype=torch.float64, grad_fn=<CopySlices>)

##Atomic Analysis
¿que procedimiento se realiza en el matlab para poder realizar la operacion matricial para la actualizacion de consecuentes a pesar de esta matriz de 0s?

In [ ]:
x_train = torch.tensor([[5., 5., 1.],
                  [6., 5., 2.],
                  [7., 3., 4.],
                  [1., 3., 5.],
                  [4., 5., 9.],
                  [4., 6., 1.]], dtype=torch.float64)
y_train = torch.tensor([5., 6., 7., 5., 9., 6.], dtype=torch.float64)

In [ ]:
x_test = torch.tensor([[1., 2., 3.],
                       [6., 8., 3.]], dtype=torch.float64)
y_test = torch.tensor([3., 8.], dtype=torch.float64)

In [ ]:
loader0 = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 8)

In [ ]:
model0 = Type3ANFIS(x_train.shape[1], init_rules = 3)

In [ ]:
model0.premises

Parameter containing:
tensor([[[ 0.1408,  0.6815],
         [-0.1948,  0.9274],
         [-0.5231,  0.7334]],

        [[-0.7663,  0.4394],
         [-0.3383,  0.0842],
         [ 0.3421,  0.6400]],

        [[ 0.1408,  0.4602],
         [-0.2977,  0.7031],
         [-0.2357,  0.7687]]], dtype=torch.float64, requires_grad=True)

In [ ]:
model0.consequents

Parameter containing:
tensor([[ 0.8437, -0.4045, -0.6241,  0.0016],
        [-0.8097,  0.3578, -0.9373, -0.0021],
        [ 0.1008,  0.5067,  0.4081,  0.2901]], dtype=torch.float64,
       requires_grad=True)

In [ ]:
new_premises = torch.tensor([[[0.3723, 0.7284],
                              [0.9371, 0.5768],
                              [0.8295, 0.0259]],

                             [[0.2068, 0.8491],
                              [0.6539, 0.3725],
                              [0.0721, 0.5932]],

                             [[0.4170, 0.4067],
                              [0.9718, 0.6669],
                              [0.9880, 0.9337]]], dtype=torch.float64)
model0.set_premises(Parameter(new_premises, requires_grad=True))

new_consequents = torch.tensor([[0.0216, 0.9394, 0.8008, 0.8840],
                                [0.5598, 0.9809, 0.8961, 0.9437],
                                [0.3008, 0.2866, 0.5975, 0.5492]], dtype=torch.float64)
new_consequents = torch.tensor([[0.9394, 0.8008, 0.8840, 0.0216],
                                [0.9809, 0.8961, 0.9437, 0.5598],
                                [0.2866, 0.5975, 0.5492, 0.3008]], dtype=torch.float64)
model0.set_consequents(Parameter(new_consequents, requires_grad=True))

In [ ]:
#---CONSEQUENTS ESTIMATION---

#needed tensors for consequents calculation
x_train = loader0.dataset.tensors[0]
y_train = loader0.dataset.tensors[1] #targets
new_consequents = torch.zeros_like(model0.consequents.data)
new_consequents2 = torch.zeros_like(model0.consequents.data)
xe = torch.cat([x_train, torch.ones(x_train.shape[0], 1)], dim=1) #extended x_train


In [ ]:
if model0.rules == 1: #if there is one rule (Nx1 tensor), torch normalization doesnt work correctly
    fls = model0.firing_levels(x_train)
    fls = fls/torch.where(fls == 0, torch.tensor(1e-6), fls)
else:
    #fls = torch.nn.functional.normalize(model0.firing_levels(x_train), p=1, dim=-1, eps=1e-12) #normalized firing_levels
    sum = torch.sum(model0.firing_levels(x_train), dim=1, keepdim=True)
    sum[sum == 0] = 1
    fls = model0.firing_levels(x_train)/sum

In [ ]:
fls

tensor([[1.0000e+00, 6.2608e-25, 0.0000e+00],
        [1.0000e+00, 2.3197e-24, 0.0000e+00],
        [4.4336e-01, 5.5664e-01, 0.0000e+00],
        [2.8206e-14, 9.9330e-01, 6.7017e-03],
        [1.3557e-42, 1.0000e+00, 0.0000e+00],
        [1.0000e+00, 1.2287e-35, 0.0000e+00]], dtype=torch.float64,
       grad_fn=<DivBackward0>)

In [ ]:
ojo = torch.tensor([[1., 0., 0.],
                    [1., 0., 0.],
                    [1., 0., 0.],
                    [0., 0., 1.],
                    [0., 1., 0.],
                    [1., 0., 0.]], dtype=torch.float64)
fls = ojo
fls

tensor([[1., 0., 0.],
        [1., 0., 0.],
        [1., 0., 0.],
        [0., 0., 1.],
        [0., 1., 0.],
        [1., 0., 0.]], dtype=torch.float64)

In [ ]:
#iterates by rule
for i in range(model0.rules):
    #firing_levels diagonal matrix
    fls_diag = torch.diag(fls[:, i])
    fls_diag2 = fls[:, i].repeat(model0.input_size + 1, 1)
    #print(fls_diag)
    #print(fls_diag @ xe)

    #new consequents
    #new_consequents[i], _, _, _ = torch.linalg.lstsq(xe.t() @ fls_diag @ xe, xe.t() @ fls_diag @ y_train)
    new_consequents[i], _, _, _ = torch.linalg.lstsq(fls_diag @ xe, fls_diag @ y_train)

    #PROBAR: new_consequents[i], _, _, _ = torch.linalg.lstsq(fls_diag @ xe, y_train)

    #print(xe.t() * fls_diag)
    new_consequents2[i], _, _, _ = torch.linalg.lstsq((xe.t() * fls_diag2).t(), y_train)

In [ ]:
new_consequents

tensor([[-0.3333,  0.6667,  1.3333,  2.0000],
        [ 0.2927,  0.3659,  0.6585,  0.0732],
        [ 0.1389,  0.4167,  0.6944,  0.1389]], dtype=torch.float64)

In [ ]:
new_consequents

tensor([[-0.3333,  0.6667,  1.3333,  2.0000],
        [ 0.2927,  0.3659,  0.6585,  0.0732],
        [ 0.1389,  0.4167,  0.6944,  0.1389]], dtype=torch.float64)

In [ ]:
new_consequents2

tensor([[-0.3333,  0.6667,  1.3333,  2.0000],
        [ 0.2927,  0.3659,  0.6585,  0.0732],
        [ 0.1389,  0.4167,  0.6944,  0.1389]], dtype=torch.float64)

In [ ]:
model0.set_consequents(Parameter(new_consequents, requires_grad=True))

In [ ]:
model0.consequents

Parameter containing:
tensor([[-0.3333,  0.6667,  1.3333,  2.0000],
        [ 0.2927,  0.3659,  0.6585,  0.0732],
        [ 0.1389,  0.4167,  0.6944,  0.1389]], dtype=torch.float64,
       requires_grad=True)

In [ ]:
pred = model0(x_train)
pred

tensor([5.0000, 6.0000, 6.3619, 4.7577, 9.0000, 6.0000], dtype=torch.float64,
       grad_fn=<SumBackward1>)

In [ ]:
mse = nn.functional.mse_loss(pred, y_train)
mse.backward()

In [ ]:
alpha = 0.01 / torch.sqrt(torch.sum(torch.pow(model0.premises.grad, 2)))
alpha

tensor(0.0005, dtype=torch.float64)

In [ ]:
current_premises = model0.premises.data
vs = current_premises[:,:,0].t()
sigmas = current_premises[:,:,1].t()
new_vs = torch.zeros_like(vs)
new_sigmas = torch.zeros_like(sigmas)

In [ ]:
for i in range(model0.rules):
    #mu
    A = 4*alpha*(1/torch.pow(sigmas[i], 2))*(x_train - vs[i])
    #wk = fls[:,i].unsqueeze(0).t()
    wk = torch.nn.functional.normalize(fls, p=1, dim=-1)[:,i].unsqueeze(0).t()
    #print(wk)
    fk = model0.outputs_by_rule(x_train)[:,i]
    zk = ((fk-pred)*(y_train-pred)).unsqueeze(0).t()

    new_vs[i] = torch.sum(A*wk*zk, dim=0)

    #sigma
    B = 4*alpha*(1/torch.pow(sigmas[i], 3))*torch.pow((x_train - vs[i]), 2)

    new_sigmas[i] = torch.sum(B*wk*zk, dim=0)

In [ ]:
current_premises[:, :, 0] += new_vs.t()
current_premises[:, :, 1] += new_sigmas.t()

In [ ]:
new_vs

tensor([[-0.0546, -0.0169, -0.0947],
        [ 0.0000,  0.0000,  0.0000],
        [-0.6119, -0.0200, -0.0111]], dtype=torch.float64,
       grad_fn=<CopySlices>)

In [ ]:
current_premises

tensor([[[ 0.3177,  0.2313],
         [ 0.9371,  0.5768],
         [ 0.2176, -4.0023]],

        [[ 0.1899,  0.7934],
         [ 0.6539,  0.3725],
         [ 0.0521,  0.4943]],

        [[ 0.3223, -0.4279],
         [ 0.9718,  0.6669],
         [ 0.9769,  0.8861]]], dtype=torch.float64, grad_fn=<CopySlices>)

In [ ]:
model0.premises

Parameter containing:
tensor([[[ 0.3177,  0.2313],
         [ 0.9371,  0.5768],
         [ 0.2176, -4.0023]],

        [[ 0.1899,  0.7934],
         [ 0.6539,  0.3725],
         [ 0.0521,  0.4943]],

        [[ 0.3223, -0.4279],
         [ 0.9718,  0.6669],
         [ 0.9769,  0.8861]]], dtype=torch.float64, requires_grad=True)